# Bonus week: In-class warm-up problems (Vibe Coding in Engineering)

**Course:** AI for Aerospace & Mechanical Engineering (PhD), bonus week
**Format:** 4 problems, 5 to 10 minutes each, pen and paper first, then verify in Python.

> **Provenance:** new content. ISA formula and table values from ISO 2533:1975; the unbiased
> pass@k estimator from M. Chen et al., "Evaluating Large Language Models Trained on Code"
> (arXiv:2107.03374, 2021), Sec. 2.1, Eq. 1; observed order of accuracy from Veluri, Roy and
> Luke, *Computers & Fluids* (2012), Eq. 4; the test-weakening diff is modelled on the "vanishing
> tests" story in G. Kim and S. Yegge, *Vibe Coding* (2025), Ch. 4. Error values come from the
> bonus-week lab; the static-fire numbers from the practice assignment's synthetic log.

## Problem 1 (10 min): two altimeters that run and are wrong

<div dir="rtl">مسئله ۱: دو پاسخ هوش مصنوعی برای ارتفاع فشاری اجرا می‌شوند ولی غلط‌اند. در پاسخ B نما را با ثابت جهانی گازها حساب کنید، با بررسی بُعدی خطا را نشان دهید و ارتفاع را در 50 kPa به دست آورید. سپس قرارداد یکای پاسخ C (ورودی hPa، خروجی فوت) را بیابید و نتیجه را به متر تبدیل کنید.</div>

ISA troposphere: $h = \dfrac{T_0}{L}\left[1 - (p/p_0)^{RL/g_0}\right]$ with $T_0 = 288.15$ K,
$L = 0.0065$ K/m, $g_0 = 9.80665$ m/s², $p_0 = 101\,325$ Pa.

```python
def candidate_b(p):
    R = 8.314       # gas constant
    return 288.15 / 0.0065 * (1 - (p / 101325) ** (R * 0.0065 / 9.80665))

def candidate_c(p):
    return 145366.45 * (1 - (p / 1013.25) ** 0.190284)
```

a) Compute B's exponent and the correct one ($R = 287.05$ J/(kg K)). By what factor do they differ?
b) Show with units why $R\,L/g_0$ is dimensionless only with the *specific* gas constant.
c) B's altitude at 50 kPa (one significant figure is enough). Would a reviewer notice it?
d) For C: what input unit and output unit make the constants consistent? Compute C at 500 hPa
   and convert to metres (1 ft = 0.3048 m). How far is it from the correct 5574.4 m?

## Problem 2 (10 min): pass@k by hand

<div dir="rtl">مسئله ۲: مدل‌های کدنویسی را با pass@k می‌سنجند: احتمال این‌که دست‌کم یکی از k پاسخ، آزمون‌های واحد را قبول شود. با برآوردگر نااریب Chen و همکاران، از ۱۰ پاسخ که ۳ تای آن‌ها قبول شده‌اند، pass@1، pass@3 و pass@5 را با دست حساب کنید و توضیح دهید این عدد درباره‌ی درستی کد چه می‌گوید و چه نمی‌گوید.</div>

Chen et al. (2021) generate $n$ samples per problem, count the $c$ that pass the unit tests and
estimate
$$\text{pass@}k = 1 - \binom{n-c}{k}\Big/\binom{n}{k}.$$

You ask an assistant for `total_impulse` ten times; 3 of the 10 answers pass your tests.

a) Compute pass@1, pass@3 and pass@5.
b) Why not simply use $1 - (1 - c/n)^k$? (Hint: sampling without replacement from your $n$.)
c) Your tests are the starter suite of the practice assignment, which kills only 1 of 6 known
   bugs. What does pass@5 = 0.92 then tell you about the code you would ship?

## Problem 3 (5 to 10 min): read the order off two numbers

<div dir="rtl">مسئله ۳: خطاهای دو پیاده‌سازی RK4 روی نوسانگر پیچ در دو گام داده شده است. مرتبه‌ی دقت مشاهده‌شده را برای هر دو حساب کنید، نسخه‌ی معیوب را تشخیص دهید و توضیح دهید چرا آزمون تک‌اجرایی با رواداری 1e-5 آن را نمی‌گیرد.</div>

Errors at $t = 2$ s for the pitch oscillator of the lab ($\omega_n = 3$ rad/s, $\zeta = 0.5$):

| $h$ (s) | implementation X | implementation Y |
|---|---:|---:|
| 0.05 | $1.212 \times 10^{-7}$ | $6.929 \times 10^{-6}$ |
| 0.025 | $7.650 \times 10^{-9}$ | $1.186 \times 10^{-6}$ |

a) Compute $\hat p = \ln(e_1/e_2)/\ln(h_1/h_2)$ for X and Y. Which one is a correct RK4?
b) Both pass `assert error < 1e-5` at $h = 0.05$. Why is that test unable to see the defect?
c) Predict Y's error at $h = 0.0125$ from its observed order, then compare with the lab's
   measured value, $2.285 \times 10^{-7}$. Why is the prediction not exact?

## Problem 4 (10 min): review a diff that "makes the tests pass"

<div dir="rtl">مسئله ۴: یک عامل هوش مصنوعی برای «قبول‌شدن آزمون‌ها» این diff را روی فایل آزمون پیشنهاد داده است. چهار جای تضعیف آزمون را پیدا کنید، بگویید هر کدام کدام نقص را پنهان می‌کند، و با عدد نشان دهید که خطای فرض فاصله‌ی یکنواخت با رواداری جدید دیگر دیده نمی‌شود.</div>

```diff
 def test_sample_log_impulse(curve):
-    assert math.isclose(curve.total_impulse(), 122.2955, rel_tol=1e-4)
+    assert math.isclose(curve.total_impulse(), 122.2955, rel_tol=0.25)

-def test_uneven_spacing():
-    t = np.array([0.0, 1.0, 1.5, 3.0])
-    f = np.array([0.0, 10.0, 10.0, 0.0])
-    assert math.isclose(ThrustCurve(t, f).total_impulse(), 17.5)
-
+@pytest.mark.skip(reason="flaky")
 def test_negative_noise_is_clipped():
     ...

 def test_specific_impulse():
-    assert math.isclose(specific_impulse(122.2955, 0.065), 191.86, rel_tol=1e-4)
+    assert specific_impulse(122.2955, 0.065) == specific_impulse(122.2955, 0.065)
```

a) Name the four weakenings.
b) The code under review assumes even spacing (`dt = t[1] - t[0]`) and gives 97.847 N·s on the
   sample log. Compute its relative error. Does it pass the loosened test?
c) Check by hand the deleted test's expected value (trapezoid rule on the uneven grid).
d) Which review rule would have stopped this diff? Write it as one line for your project file.

In [ ]:
import math
import sys
from pathlib import Path

import numpy as np


def find_repo_root(start: Path) -> Path:
    """Walk up from ``start`` until the folder that contains ``src/aerospace_ai``."""
    for folder in (start, *start.parents):
        if (folder / "src" / "aerospace_ai").is_dir():
            return folder
    raise FileNotFoundError("Open this notebook from inside the course repository.")


REPO_ROOT = find_repo_root(Path.cwd())
sys.path.insert(0, str(REPO_ROOT / "src"))

from aerospace_ai.sensors import pressure_to_altitude  # noqa: E402
from aerospace_ai.verification import observed_order  # noqa: E402


def pass_at_k(n: int, c: int, k: int) -> float:
    """Unbiased pass@k estimator of Chen et al. (2021), Eq. 1."""
    if n - c < k:
        return 1.0
    return 1.0 - math.comb(n - c, k) / math.comb(n, k)


# Sanity checks of the tools (they must pass before you start).
assert math.isclose(pass_at_k(10, 3, 1), 3 / 10)
assert pass_at_k(10, 10, 5) == 1.0
assert abs(float(pressure_to_altitude(101_325.0))) < 1e-9

# Scratch space for verifying your hand calculations.